In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from train.models import FullModel
from util import image as u_image
from util import dataset as u_dataset

In [ ]:
def camera_from_label(label):
    """Calculate the camera roll pitch and height from the camera pose in the data.

    Args:
        label: the label with the camera pose

    Returns:
        A tuple of roll, pitch and height.
    """
    alpha = np.arccos(label["cpose"]["z"][2])
    if np.abs(alpha) < 0.01:
        roll = pitch = 0
    else:
        sin_alpha = np.sqrt(1 - label["cpose"]["z"][2]*label["cpose"]["z"][2])
        roll = -label["cpose"]["z"][1] / sin_alpha * alpha
        pitch = label["cpose"]["z"][0] / sin_alpha * alpha
    height = label["cpose"]["h"] * 0.001
    return (roll, pitch, height)

In [ ]:
def intrinsics_from_label(label):
    """
    Get the camera intrinsics from the label.
    
    Args:
        label: A label from the dataset
        
    Returns:
        The camera intrinsics as a tuple (cx, cy, fx, fy).
    """
    
    return (label["cintr"]["cx"], label["cintr"]["cy"], label["cintr"]["fx"], label["cintr"]["fy"])

In [ ]:
def get_dataset(directory):
    # Load the dataset 
    #TODO: this only loads the first 100 images!!!
    labels = u_dataset.load_labels(directory)[:100]

    images = []
    cameras = []
    intrinsics = []

    for label in labels:
        # Load the image for the label in YUYV format
        images.append(u_dataset.load_image(directory, label, image_format=u_image.ImageFormat.YUYV))
        # Load the camera pose for the label (roll, pitch, height)
        cameras.append(camera_from_label(label))
        # Load the camera intrinsics for the label
        intrinsics.append(intrinsics_from_label(label))

    # Combine the images, cameras and intrinsics into a single tensorflow dataset
    return tf.data.Dataset.from_tensor_slices(
        {
            "image": tf.reshape(tf.constant(images), (-1, 480, 320, 4)),
            "camera": tf.constant(cameras, dtype=tf.float32),
            "intrinsics": tf.constant(intrinsics, dtype=tf.float32),
        }
    )

In [ ]:
train_ds = get_dataset("/Users/laurensschiefelbein/Developer/MA_LabelingTool/data/Joerg_Joerg_CompetitionWalk_GO2025__HULKs_1stHalf_5")

train_ds = train_ds.shuffle(32)
train_ds = train_ds.batch(32)

# Upper camera dimensions. Width is halved because of YUYV format
model = FullModel(480, 320) 
model.compile(optimizer=tf.keras.optimizers.Adam())
model.fit(x=train_ds, epochs=10)

print(model.summary())



# Save model
# model.save("./models/model.keras")

In [ ]:
test_directory = "/Users/laurensschiefelbein/Developer/MA_LabelingTool/data/Joerg_Joerg_CompetitionWalk_GO2025__HULKs_1stHalf_5"

label = u_dataset.load_labels(test_directory)[101]

image = u_dataset.load_image(test_directory, label, image_format=u_image.ImageFormat.YUYV)
camera = camera_from_label(label)
intrinsics = intrinsics_from_label(label)

image_yuv = tf.reshape(tf.constant(image), (-1, 480, 320, 4))
camera = tf.constant(camera, dtype=tf.float32)
intrinsics = tf.constant(intrinsics, dtype=tf.float32)

print("Image: ", image_yuv.shape)
print("Camera: ", camera)
print("Intrinsics: ", intrinsics)




results = model((image_yuv, camera, intrinsics), training=None)



In [ ]:
print(results["ball"][0].shape) # 0 -> patch, 1 -> masks

# Patch Shape: [B, candidate, height, width, channels]

print(results["ball"][1]) # 0 -> patch, 1 -> masks

print(image.shape)

image_converted = tf.reshape((image_yuv), (-1, 480, 640, 2))
# Plot the image
# plt.imshow(image_yuv[0, ...].numpy() / 255 ,)
plt.imshow(image_converted[0, ..., 0] / 255)
plt.title("Image")

print(results["ball"][0][0, 0, ...].numpy() / 255)

fig, axes = plt.subplots(5)
# axes[0].imshow(results["ball"][0][0, 0, ...].numpy() / 255)
# axes[1].imshow(results["ball"][0][0, 1, ...].numpy() / 255)
# axes[2].imshow(results["ball"][0][0, 2, ...].numpy() / 255)
# axes[3].imshow(results["ball"][0][0, 3, ...].numpy() / 255)
# axes[4].imshow(results["ball"][0][0, 4, ...].numpy() / 255)

print(results["ball"][0][0, 1, ...].shape)
print(np.reshape(results["ball"][0][0, 1, ...].numpy()[:, :, [0, 2]], (32, 64, 1)).shape)
# print(tf.reshape(results["ball"][0][0, 1, : ,: , [0,2]], (32, 64, 2)).shape)

axes[0].imshow(results["ball"][0][0, 1, ...].numpy()[:, :, [0, 1, 3]] / 255)
axes[1].imshow(results["ball"][0][0, 2, ...].numpy()[:, :, [0, 1, 3]] / 255)
axes[2].imshow(results["ball"][0][0, 3, ...].numpy()[:, :, 1:4] / 255)

plt.show()

In [ ]:
print(model.encoder.output_shape)
print(model.encoder.input_shape)

print(model.categories["ball"])